<a href="https://colab.research.google.com/github/guitorte/audio/blob/claude/song-to-midi-converter-KRClC/Song_to_MIDI_Converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 Song to MIDI Converter

Convert any audio file (MP3, WAV, M4A, FLAC…) to a playable **MIDI file** using
[Spotify's Basic Pitch](https://github.com/spotify/basic-pitch) — a neural network for
automatic music transcription.

## Workflow
| Step | Action |
|------|--------|
| 1 | **Install** – run once, restart runtime when prompted |
| 2 | **Load libraries** |
| 3 | *(Optional)* Mount Google Drive |
| 4 | **Upload** your audio file |
| 5 | **Analyse** – waveform, spectrogram, BPM, key |
| 6 | **Convert** audio → MIDI |
| 7 | **Visualise** the piano roll |
| 8 | **Preview** MIDI playback |
| 9 | **Download** the MIDI file |
| 10 | *(Optional)* Transpose / quantise |


In [ ]:
# @title 1️⃣  Install Dependencies  ← Run this first, then **Restart Runtime**

# Python packages
!pip install -q basic-pitch onnxruntime mir_eval librosa pretty-midi \
                soundfile matplotlib mido importlib_resources

# System audio synth (for MIDI playback)
!apt-get install -y -q fluidsynth 2>/dev/null

print()
print('=' * 55)
print('✅  Installation complete!')
print('⚠️   Go to  Runtime ▸ Restart Session  then run cell 2 onwards.')
print('=' * 55)


In [ ]:
# @title 2️⃣  Load Libraries

import importlib, importlib.util, os, subprocess, warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Audio, display, FileLink

warnings.filterwarnings('ignore')

# -- lazy-load packages that may need a post-install restart --
def _require(pkg, pip_name=None):
    importlib.invalidate_caches()
    try:
        return importlib.import_module(pkg)
    except ImportError:
        label = pip_name or pkg.replace('_', '-')
        raise SystemExit(
            f'\n❌  Cannot import "{pkg}".\n'
            f'   Run Cell 1, then go to Runtime ▸ Restart Session.\n'
        )

librosa     = _require('librosa')
pretty_midi = _require('pretty_midi', 'pretty-midi')
mido        = _require('mido')

# Basic Pitch
try:
    from basic_pitch.inference import predict as bp_predict
except ImportError:
    raise SystemExit(
        '\n❌  Cannot import basic_pitch.\n'
        '   Run Cell 1, then go to Runtime ▸ Restart Session.\n'
    )

# Colab flag
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

OUTPUT_DIR = '/content/midi_outputs' if IN_COLAB else '/tmp/midi_outputs'
SAVE_DIR   = OUTPUT_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅  All libraries loaded.')
print(f'   Running in Colab: {IN_COLAB}')
print(f'   Output directory: {OUTPUT_DIR}')


In [ ]:
# @title 3️⃣  Mount Google Drive (optional)

mount_drive = False  # @param {type:"boolean"}

SAVE_DIR = OUTPUT_DIR
if mount_drive and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    SAVE_DIR = '/content/drive/MyDrive/MIDI_Outputs'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f'✅  Google Drive mounted.  Files → {SAVE_DIR}')
elif mount_drive:
    print('ℹ️  Not in Colab — Google Drive mount skipped.')
else:
    print(f'ℹ️  Saving to local dir: {SAVE_DIR}')


In [ ]:
# @title 4️⃣  Load Audio File

audio_source = 'upload'  # @param ["upload", "url", "example"]
url_input    = ''        # @param {type:"string"}

audio_file = None

if audio_source == 'upload':
    if IN_COLAB:
        from google.colab import files as _gf
        print('📁  Select your audio file (MP3, WAV, M4A, FLAC…)')
        uploaded = _gf.upload()
        if uploaded:
            fname = list(uploaded.keys())[0]
            audio_file = os.path.join(OUTPUT_DIR, fname)
            with open(audio_file, 'wb') as fh:
                fh.write(uploaded[fname])
            print(f'✅  Uploaded: {fname}')
        else:
            print('⚠️  No file selected.')
    else:
        audio_file = input('Path to audio file: ').strip()
        if not os.path.exists(audio_file):
            print(f'❌  File not found: {audio_file}')
            audio_file = None

elif audio_source == 'url':
    if url_input.strip():
        import urllib.request
        ext = url_input.rsplit('.', 1)[-1].split('?')[0] or 'mp3'
        audio_file = os.path.join(OUTPUT_DIR, f'downloaded.{ext}')
        print(f'⬇️  Downloading {url_input} …')
        urllib.request.urlretrieve(url_input, audio_file)
        print(f'✅  Saved to {audio_file}')
    else:
        print('⚠️  Paste a URL in the url_input field above.')

elif audio_source == 'example':
    import scipy.io.wavfile as wavfile
    audio_file = os.path.join(OUTPUT_DIR, 'example.wav')
    sr_ex = 22050
    t_ex  = np.linspace(0, 10, sr_ex * 10)
    # Simple Do-Re-Mi-Fa-Sol melody
    freqs  = [261.63, 293.66, 329.63, 349.23, 392.00]
    sig    = np.zeros_like(t_ex)
    nd     = sr_ex * 2
    for i, f in enumerate(freqs):
        seg = np.sin(2 * np.pi * f * t_ex[i*nd:(i+1)*nd])
        # Add slight decay envelope
        env = np.exp(-np.linspace(0, 3, len(seg)))
        sig[i*nd:(i+1)*nd] = 0.4 * seg * env
    wavfile.write(audio_file, sr_ex, (sig * 32767).astype(np.int16))
    print(f'✅  Example audio created: {audio_file}')

if audio_file and os.path.exists(audio_file):
    mb = os.path.getsize(audio_file) / 1024**2
    print(f'📊  Size: {mb:.2f} MB  |  Path: {audio_file}')
    print('🔊  Original audio:')
    display(Audio(audio_file))


In [ ]:
# @title 5️⃣  Analyse Audio

if not audio_file or not os.path.exists(audio_file):
    print('⚠️  Load an audio file first (Cell 4).')
else:
    print(f'Loading: {os.path.basename(audio_file)} …')
    y, sr = librosa.load(audio_file, sr=None)
    dur   = librosa.get_duration(y=y, sr=sr)

    # Tempo
    tempo_arr, _ = librosa.beat.beat_track(y=y, sr=sr)
    tempo = float(np.atleast_1d(tempo_arr)[0])

    # Key via chroma
    chroma  = librosa.feature.chroma_stft(y=y, sr=sr)
    key_idx = int(np.argmax(np.mean(chroma, axis=1)))
    KEY_NAMES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
    key = KEY_NAMES[key_idx]

    # Spectral centroid (brightness)
    sc   = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    rms  = librosa.feature.rms(y=y)[0]

    print(f'\n📊  Audio Analysis')
    print(f'  Duration  : {dur:.1f} s')
    print(f'  Tempo     : {tempo:.0f} BPM')
    print(f'  Key       : {key}')
    print(f'  Sample rate: {sr} Hz')
    print(f'  Avg RMS   : {np.mean(rms):.4f}')
    print(f'  Brightness: {np.mean(sc):.0f} Hz')

    # ---- plots ----
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))

    # Waveform
    librosa.display.waveshow(y, sr=sr, ax=axes[0], color='steelblue')
    axes[0].set_title('Waveform', fontweight='bold')

    # Spectrogram
    D   = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    img = librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='log',
                                   ax=axes[1], cmap='magma')
    axes[1].set_title('Spectrogram', fontweight='bold')
    fig.colorbar(img, ax=axes[1], format='%+2.0f dB')

    # Chroma
    ch_img = librosa.display.specshow(chroma, sr=sr, x_axis='time',
                                      y_axis='chroma', ax=axes[2], cmap='coolwarm')
    axes[2].set_title('Chromagram', fontweight='bold')
    fig.colorbar(ch_img, ax=axes[2])

    plt.tight_layout()
    analysis_png = os.path.join(OUTPUT_DIR, 'audio_analysis.png')
    plt.savefig(analysis_png, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'\n✅  Analysis complete.')


In [ ]:
# @title 6️⃣  Convert Audio → MIDI

if not audio_file or not os.path.exists(audio_file):
    print('⚠️  Load an audio file first (Cell 4).')
else:
    print(f'⚙️  Transcribing {os.path.basename(audio_file)} …')
    print('   (1–3 min for a full song — GPU speeds this up considerably)')

    try:
        model_output, midi_data, note_events = bp_predict(audio_file)

        n_notes = len(note_events)
        n_instr = len(midi_data.instruments)
        midi_dur = midi_data.get_end_time()

        print(f'\n✅  Transcription complete!')
        print(f'  Notes detected : {n_notes}')
        print(f'  MIDI tracks    : {n_instr}')
        print(f'  MIDI duration  : {midi_dur:.1f} s')

        if n_instr > 0 and midi_data.instruments[0].notes:
            notes    = midi_data.instruments[0].notes
            pitches  = [n.pitch for n in notes]
            durs_n   = [n.end - n.start for n in notes]
            vels     = [n.velocity for n in notes]
            print(f'  Pitch range    : {min(pitches)} – {max(pitches)}')
            print(f'  Avg note dur   : {np.mean(durs_n):.3f} s')
            print(f'  Velocity range : {min(vels)} – {max(vels)}')

        midi_path = os.path.join(SAVE_DIR, 'transcription.mid')
        midi_data.write(midi_path)
        print(f'\n💾  MIDI saved: {midi_path}')

    except Exception as err:
        import traceback
        print(f'❌  Error: {err}')
        traceback.print_exc()


In [ ]:
# @title 7️⃣  Visualise MIDI (Piano Roll)

if 'midi_data' not in dir() or midi_data is None:
    print('⚠️  Convert audio first (Cell 6).')
else:
    # Piano roll
    piano_roll = midi_data.get_piano_roll(fs=10)
    pitch_min, pitch_max = 28, 96
    roll = piano_roll[pitch_min:pitch_max + 1, :]

    KEY_NAMES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']

    fig, ax = plt.subplots(figsize=(16, 6))
    extent = [0, roll.shape[1] / 10, pitch_min, pitch_max]
    ax.imshow(roll, aspect='auto', origin='lower',
              cmap='Blues', interpolation='nearest', extent=extent)
    ax.set_xlabel('Time (seconds)', fontsize=11)
    ax.set_ylabel('MIDI Pitch', fontsize=11)
    ax.set_title('Piano Roll — Transcribed Melody', fontsize=13, fontweight='bold')

    yticks = range(pitch_min, pitch_max + 1, 4)
    ax.set_yticks(list(yticks))
    ax.set_yticklabels([f"{KEY_NAMES[p % 12]}{p // 12}" for p in yticks], fontsize=7)

    plt.colorbar(ax.images[0], ax=ax, label='Velocity')
    plt.tight_layout()
    piano_roll_png = os.path.join(OUTPUT_DIR, 'piano_roll.png')
    plt.savefig(piano_roll_png, dpi=100, bbox_inches='tight')
    plt.show()

    # Distribution plots
    notes    = midi_data.instruments[0].notes
    pitches  = [n.pitch for n in notes]
    durs_n   = [n.end - n.start for n in notes]

    fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    ax1.hist(pitches, bins=24, color='steelblue', edgecolor='white', linewidth=0.5)
    ax1.set_xlabel('MIDI Pitch')
    ax1.set_ylabel('Count')
    ax1.set_title('Pitch Distribution')
    ax1.grid(alpha=0.3)

    ax2.hist(durs_n, bins=24, color='seagreen', edgecolor='white', linewidth=0.5)
    ax2.set_xlabel('Note Duration (s)')
    ax2.set_ylabel('Count')
    ax2.set_title('Note Duration Distribution')
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    dist_png = os.path.join(OUTPUT_DIR, 'note_distribution.png')
    plt.savefig(dist_png, dpi=100, bbox_inches='tight')
    plt.show()
    print('✅  Visualisation complete.')


In [ ]:
# @title 8️⃣  MIDI Playback Preview

if 'midi_data' not in dir() or midi_data is None:
    print('⚠️  Convert audio first (Cell 6).')
else:
    midi_path  = os.path.join(SAVE_DIR, 'transcription.mid')
    playback_wav = os.path.join(OUTPUT_DIR, 'playback.wav')
    sf2_candidates = [
        '/usr/share/sounds/sf2/FluidR3_GM.sf2',
        '/usr/share/sounds/sf2/TimGM6mb.sf2',
    ]
    sf2_path = next((p for p in sf2_candidates if os.path.exists(p)), None)

    synthesised = False
    if sf2_path:
        try:
            r = subprocess.run(
                ['fluidsynth', '-ni', sf2_path, midi_path, '-F', playback_wav, '-r', '22050'],
                capture_output=True, timeout=120
            )
            if os.path.exists(playback_wav) and os.path.getsize(playback_wav) > 1024:
                synthesised = True
                print('✅  Synthesised with FluidSynth.')
        except Exception as fe:
            print(f'ℹ️  FluidSynth error: {fe}')

    if not synthesised:
        print('ℹ️  FluidSynth not available — using sine-wave fallback.')
        sr_out = 22050
        total  = midi_data.get_end_time()
        sig    = np.zeros(int(sr_out * total))
        for note in midi_data.instruments[0].notes:
            s = int(note.start * sr_out)
            e = int(note.end   * sr_out)
            f = 440.0 * (2 ** ((note.pitch - 69) / 12))
            t = np.arange(e - s) / sr_out
            env = np.exp(-3 * t / max(e - s, 1) * sr_out)  # decay
            sig[s:e] += (note.velocity / 127.0) * 0.3 * np.sin(2 * np.pi * f * t) * env
        peak = np.max(np.abs(sig))
        if peak > 0:
            sig /= peak
        display(Audio(data=sig, rate=sr_out))
    else:
        display(Audio(playback_wav))


In [ ]:
# @title 9️⃣  Download Results

if 'midi_data' not in dir() or midi_data is None:
    print('⚠️  No MIDI generated yet.')
else:
    artefacts = [
        ('transcription.mid',     SAVE_DIR,   '🎹 MIDI file'),
        ('audio_analysis.png',    OUTPUT_DIR, '📈 Audio analysis'),
        ('piano_roll.png',        OUTPUT_DIR, '🎼 Piano roll'),
        ('note_distribution.png', OUTPUT_DIR, '📊 Note distribution'),
    ]

    print('📥  Available files:\n')
    for fname, fdir, label in artefacts:
        fpath = os.path.join(fdir, fname)
        if os.path.exists(fpath):
            kb = os.path.getsize(fpath) / 1024
            print(f'  {label}: {fname} ({kb:.1f} KB)')

    print()
    if IN_COLAB:
        from google.colab import files as _gf
        midi_out = os.path.join(SAVE_DIR, 'transcription.mid')
        if os.path.exists(midi_out):
            print('⬇️  Downloading MIDI file to your computer…')
            _gf.download(midi_out)
    else:
        midi_out = os.path.join(SAVE_DIR, 'transcription.mid')
        print(f'MIDI path: {midi_out}')
        display(FileLink(midi_out))


In [ ]:
# @title 🔟  Advanced: Transpose & Quantise (optional)

transpose_semitones = 0    # @param {type:"slider", min:-12, max:12, step:1}
quantize_grid_ms    = 100  # @param {type:"slider", min:25, max:500, step:25}

if 'midi_data' not in dir() or midi_data is None:
    print('⚠️  No MIDI to process.')
else:
    grid = quantize_grid_ms / 1000

    out_midi = pretty_midi.PrettyMIDI()
    for instr in midi_data.instruments:
        new_instr = pretty_midi.Instrument(program=instr.program, name=instr.name)
        for note in instr.notes:
            new_pitch = int(np.clip(note.pitch + transpose_semitones, 0, 127))
            new_start = round(note.start / grid) * grid
            new_end   = round(note.end   / grid) * grid
            if new_end <= new_start:
                new_end = new_start + grid
            new_instr.notes.append(
                pretty_midi.Note(
                    velocity=note.velocity,
                    pitch=new_pitch,
                    start=new_start,
                    end=new_end,
                )
            )
        out_midi.instruments.append(new_instr)

    suffix  = ''
    if transpose_semitones:
        suffix += f'_T{transpose_semitones:+d}'
    suffix += f'_Q{quantize_grid_ms}ms'
    out_path = os.path.join(SAVE_DIR, f'transcription{suffix}.mid')
    out_midi.write(out_path)

    print(f'✅  Saved: {os.path.basename(out_path)}')
    print(f'   Transpose : {transpose_semitones:+d} semitones')
    print(f'   Quantise  : {quantize_grid_ms} ms grid')

    if IN_COLAB:
        from google.colab import files as _gf
        _gf.download(out_path)


---

## 📚 Notes & Tips

### How Basic Pitch Works
1. Audio is resampled to 22 050 Hz and split into short frames
2. A lightweight CNN predicts note onset, pitch activation, and contour for each frame
3. A post-processing step converts frame predictions into discrete MIDI notes

### Best Results
| Audio type | Expected quality |
|------------|------------------|
| Solo instrument / clean melody | ⭐⭐⭐⭐⭐ |
| Lead vocal + simple backing | ⭐⭐⭐⭐ |
| Full band / complex harmony | ⭐⭐⭐ |
| Noisy / heavily compressed audio | ⭐⭐ |

### Tips
- Use a **GPU runtime** (`Runtime ▸ Change runtime type ▸ T4 GPU`) for much faster transcription
- **Trim** the audio to the interesting section before uploading
- **Normalise** loudness if the recording is very quiet
- Open the MIDI in a DAW (Ableton, FL Studio, GarageBand, Reaper…) to edit

---
**Powered by [Spotify Basic Pitch](https://github.com/spotify/basic-pitch)**
